# 02 — Expression clustering and marker candidates

PCA and Leiden use expression only. Cluster markers use the preserved lognorm layer. Cluster p-values are exploratory, not tests of rejection between patients.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from src import preprocessing, utils, visualization
from src.spatial import build_spatial_graph, validate_graph, spatial_lr
settings = utils.load_settings()
params = settings['params']
seed = params['seed']
sc.settings.set_figure_params(dpi=params['visualization']['figure_dpi'], facecolor='white')

In [ ]:
run = utils.begin_stage(settings, 'adata_clustered.h5ad', 'adata_qc.h5ad')
adata = sc.read_h5ad(run / 'processed' / 'adata_qc.h5ad')
cfg = params['clustering']
embedding = adata[:, adata.var['highly_variable']].copy()
sc.pp.scale(embedding, max_value=10)
n_pcs = min(cfg['n_pcs'], embedding.n_vars - 1, embedding.n_obs - 1)
if n_pcs < 2:
    raise ValueError('Too few variable genes/spots for PCA')
sc.tl.pca(embedding, n_comps=n_pcs, random_state=seed)
adata.obsm['X_pca'] = embedding.obsm['X_pca'].copy()
del embedding
sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=cfg['n_neighbors'], random_state=seed)
sc.tl.leiden(adata, resolution=cfg['resolution'], random_state=seed, flavor='igraph', directed=False, n_iterations=2)
sc.tl.umap(adata, random_state=seed)
sc.tl.rank_genes_groups(adata, groupby='leiden', layer='lognorm', use_raw=False, method='wilcoxon', n_genes=cfg['top_markers'])
markers = sc.get.rank_genes_groups_df(adata, group=None)
if not np.isfinite(markers['logfoldchanges']).all():
    raise ValueError('Marker fold changes are not finite')
markers.to_csv(run / 'tables' / 'cluster_markers.csv', index=False)

In [ ]:
sp = params['spatial']
build_spatial_graph(adata, n_neighs=sp['n_neighs'], coord_type=sp['coord_type'])
validate_graph(adata)
sc.pl.umap(adata, color=['leiden', 'sample'], show=False)
plt.savefig(run / 'figures' / 'expression_umap.png', bbox_inches='tight')
plt.show()
visualization.plot_spatial_clusters(adata, save_path=run / 'figures' / 'expression_clusters_on_tissue.png', spot_size=params['visualization']['spot_size'])
plt.show()
utils.save_checkpoint(adata, run / 'processed' / 'adata_clustered.h5ad')